# Aplicações Práticas de Validação no Ciclo de Análise de Dados

## 1. Validação em Pipelines de Dados

In [ ]:
import pandas as pd
import numpy as np
import pandera.pandas as pa
from datetime import datetime

# Definindo esquema de validação
schema = pa.DataFrameSchema({
    'id': pa.Column(int, checks=pa.Check.gt(0)),
    'data': pa.Column('datetime64[ns]', checks=pa.Check.le(datetime.now())),
    'valor': pa.Column(float, checks=pa.Check.gt(0)),
    'categoria': pa.Column(str, checks=pa.Check.isin(['A', 'B', 'C'])),
    'status': pa.Column(str, checks=pa.Check.isin(['ativo', 'inativo']))
})

def pipeline_etl():
    # Extração
    df = pd.read_csv('dados.csv')
    
    # Transformação
    df['data'] = pd.to_datetime(df['data'])
    df['valor'] = df['valor'].astype(float)
    
    # Validação
    try:
        schema.validate(df)
        print("Dados validados com sucesso!")
        
        # Carregamento
        df.to_parquet('dados_validados.parquet')
        return True
    except pa.errors.SchemaError as e:
        print(f"Erro na validação: {e}")
        return False

## 2. Validação em APIs de Dados

In [ ]:
pip install fastapi

In [ ]:
from fastapi import FastAPI, HTTPException
import numpy as np
import pandera.pandas as pa
import pandas as pd
from pydantic import BaseModel

app = FastAPI()

# Definindo esquema de validação
schema = pa.DataFrameSchema({
    'nome': pa.Column(str, checks=pa.Check.str_length(min_value=3, max_value=100)),
    'idade': pa.Column(int, checks=pa.Check.ge(0)),
    'email': pa.Column(str, checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'))
})

class Usuario(BaseModel):
    nome: str
    idade: int
    email: str

@app.post("/usuarios")
async def criar_usuario(usuario: Usuario):
    # Convertendo para DataFrame
    df = pd.DataFrame([usuario.dict()])
    
    # Validando dados
    try:
        schema.validate(df)
        # Processar usuário...
        return {"mensagem": "Usuário criado com sucesso"}
    except pa.errors.SchemaError as e:
        raise HTTPException(status_code=400, detail=str(e))

## 3. Validação em Análise de Dados

In [ ]:
import pandas as pd
import numpy as np
import pandera.pandas as pa


class AnalisadorVendas:
    def __init__(self):
        self.schema = pa.DataFrameSchema({
            'data': pa.Column('datetime64[ns]'),
            'produto': pa.Column(str),
            'quantidade': pa.Column(int, checks=pa.Check.gt(0)),
            'valor': pa.Column(float, checks=pa.Check.gt(0))
        })
    
    def validar_dados(self, df):
        try:
            self.schema.validate(df)
            return True
        except pa.errors.SchemaError as e:
            print(f"Erro na validação: {e}")
            return False
    
    def gerar_relatorio(self, df):
        if not self.validar_dados(df):
            return None
        
        # Cálculos do relatório
        total_vendas = df['valor'].sum()
        media_quantidade = df['quantidade'].mean()
        produtos_unicos = df['produto'].nunique()
        
        return {
            'total_vendas': total_vendas,
            'media_quantidade': media_quantidade,
            'produtos_unicos': produtos_unicos
        }

## 4. Validação em Machine Learning

In [ ]:
pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import pandera.pandas as pa
from sklearn.ensemble import RandomForestClassifier

class PipelineML:
    def __init__(self):
        self.schema = pa.DataFrameSchema({
            'idade': pa.Column(float, checks=pa.Check.ge(0)),
            'renda': pa.Column(float, checks=pa.Check.gt(0)),
            'escolaridade': pa.Column(str, checks=pa.Check.isin(['fundamental', 'medio', 'superior'])),
            'target': pa.Column(int, checks=pa.Check.isin([0, 1]))
        })
        self.modelo = RandomForestClassifier()
    
    def validar_dados(self, df):
        try:
            self.schema.validate(df)
            return True
        except pa.errors.SchemaError as e:
            print(f"Erro na validação: {e}")
            return False
    
    def preparar_dados(self, df):
        # Codificação de variáveis categóricas
        df['escolaridade'] = pd.Categorical(df['escolaridade']).codes
        return df
    
    def treinar(self, df):
        if not self.validar_dados(df):
            return None
        
        df = self.preparar_dados(df)
        X = df.drop('target', axis=1)
        y = df['target']
        
        self.modelo.fit(X, y)
        return self.modelo

## 5. Validação em Monitoramento de Dados

In [ ]:
import pandas as pd
import numpy as np
import pandera.pandas as pa
from datetime import datetime, timedelta

class MonitorDados:
    def __init__(self):
        self.schema = pa.DataFrameSchema({
            'timestamp': pa.Column('datetime64[ns]'),
            'metrica': pa.Column(str),
            'valor': pa.Column(float)
        })
    
    def validar_dados(self, df):
        try:
            self.schema.validate(df)
            return True
        except pa.errors.SchemaError as e:
            print(f"Erro na validação: {e}")
            return False
    
    def verificar_tendencia(self, df):
        if not self.validar_dados(df):
            return None
        
        # Análise de tendência
        df['media_movel'] = df['valor'].rolling(window=7).mean()
        tendencia = df['valor'].iloc[-1] > df['media_movel'].iloc[-1]
        
        return {
            'tendencia': 'alta' if tendencia else 'baixa',
            'ultimo_valor': df['valor'].iloc[-1],
            'media_movel': df['media_movel'].iloc[-1]
        }
    
    def verificar_anomalias(self, df):
        if not self.validar_dados(df):
            return None
        
        # Detecção de anomalias
        media = df['valor'].mean()
        std = df['valor'].std()
        limite = media + 3 * std
        
        anomalias = df[df['valor'] > limite]
        return anomalias

# Boas Práticas para Implementação
Validação em Múltiplas Camadas: Implemente validação em diferentes pontos do pipeline
    
Logs Detalhados: Registre todas as validações e correções realizadas
    
Tratamento de Erros: Defina estratégias claras para lidar com dados inválidos
    
Documentação: Mantenha documentação atualizada dos esquemas de validação
    
Testes: Implemente testes para os esquemas de validação
    
Versionamento: Controle versões dos esquemas de validação
    
Monitoramento Contínuo: Implemente alertas para mudanças nos padrões dos dados